# CNN / LeNet5 reference evaluation

Loads two `.pt` checkpoints -- one produced by
`sazz/gpu_friendly/scripts/cnn_reference.py` (small CNN, `D=19466`), one by
`sazz/gpu_friendly/scripts/lenet_reference.py` (LeNet5, `D=61706`) -- and
inspects both side by side before either gets used as `--map-path` for a
real sampler run in `mnist_cnn_grid.py`.

Point the `CNN_CKPT_PATH`/`LENET_CKPT_PATH` cell below at any checkpoints on
disk -- this notebook handles both the current schema and older/schema-stale
checkpoints (missing `activation`/`pool`/`fan_in_scaling`/`sigma_inv_source`/
`map_history`/`architecture` keys) gracefully via `.get(...)` fallbacks,
matching this repo's other `grid_*` notebooks' convention.

`mnist_cnn_grid.py`'s `build_target` is hardcoded to `CNN` (no LeNet5
dispatch), so this notebook rebuilds `BayesianModule` itself for both
architectures rather than going through `build_target` -- picking the
`nn.Module` class from the checkpoint's own `architecture` field (defaults
to `"cnn"` for older checkpoints that lack it, since `cnn_reference.py`'s
own checkpoints were never retrofitted with the field). This is the same
"trust the checkpoint's own stored fields over the current script defaults"
approach the notebook already used for schema-stale checkpoints, just
applied one layer earlier (architecture choice, not only config values).

In [ ]:
import os
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
%matplotlib inline

if Path.cwd().name == "notebooks":
    os.chdir("..")

torch.set_default_dtype(torch.float64)

plt.rcParams.update({
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.grid":          True,
    "grid.alpha":         0.3,
    "font.size":          11,
})

from sazz.gpu_friendly.scripts.mnist_cnn_grid import (
    load_mnist_subset, CNNConfig, print_preactivation_diagnostic,
    DTYPE, DEVICE,
)
from sazz.gpu_friendly.scripts.cnn_reference import eval_accuracy
from sazz.gpu_friendly.models.neural_networks import CNN, LeNet5
from sazz.gpu_friendly.models.model import BayesianModule
from sazz.gpu_friendly.models.priors import build_fan_in_prior_precision, build_kappa_from_inclusion

ARCHITECTURES = {"cnn": CNN, "lenet5": LeNet5}

print(f"device={DEVICE}  dtype={DTYPE}")

## 1. Load checkpoints + rebuild targets

`load_reference` below rebuilds `BayesianModule` directly (`CNN` or
`LeNet5`, dispatched on the checkpoint's own `architecture` field) rather
than calling `mnist_cnn_grid.py`'s `build_target`, which is hardcoded to
`CNN` only. It still reuses `build_fan_in_prior_precision` -- the same
prior-construction helper `build_target` itself calls -- so the only thing
this notebook does differently is choosing which `nn.Module` class to
instantiate; the prior/likelihood construction is identical to what a real
sampler run would do.

Caveat: the checkpoint does not store the `--seed` its producing script was
run with (only `n_train`/`n_steps`) -- this reconstructs the training
subset using `seed=0`, both `cnn_reference.py`'s and `lenet_reference.py`'s
own CLI default. If a checkpoint was produced with a non-default `--seed`,
the reconstructed `X_train`/`y_train` below will not be bit-identical to
what it was actually fit on (structurally the same MNIST subset-selection
procedure, just a different random subset) -- only relevant for anything
that depends on exact per-example values, not for the aggregate diagnostics
this notebook computes.

In [ ]:
# --- Point these at any checkpoints produced by cnn_reference.py / lenet_reference.py ---
CNN_CKPT_PATH = Path("results/maps/cnn_reference_N60000_steps10000.pt")
LENET_CKPT_PATH = Path("results/maps/lenet_reference_N60000_steps10000.pt")
SEED = 0  # both scripts' own CLI default; see caveat above


def load_reference(ckpt_path: Path, seed: int) -> dict:
    ckpt = torch.load(ckpt_path, weights_only=False)
    architecture = ckpt.get("architecture", "cnn")
    module_cls = ARCHITECTURES[architecture]

    print(f"=== {ckpt_path.name}  (architecture={architecture}) ===")
    schema_fields = [
        "D", "activation", "pool", "fan_in_scaling", "prior_std_weight",
        "prior_std_bias", "sigma_inv_source", "train_acc", "test_acc",
        "n_train", "n_steps",
    ]
    for f in schema_fields:
        print(f"  {f:<18} = {ckpt.get(f, 'n/a')}")
    has_history = "map_history" in ckpt
    print(f"  {'map_history':<18} = {'present (' + str(len(ckpt['map_history'])) + ' entries)' if has_history else 'n/a (older checkpoint)'}")

    data = load_mnist_subset(
        n_train=ckpt["n_train"], n_test=500, seed=seed,
        data_dir=Path("datasets"), dtype=DTYPE, device=DEVICE,
    )

    cfg = CNNConfig(
        activation=ckpt.get("activation", "tanh"),
        pool=ckpt.get("pool", "avg"),
        prior_std_weight=ckpt["prior_std_weight"],
        prior_std_bias=ckpt["prior_std_bias"],
        fan_in_scaling=ckpt.get("fan_in_scaling", True),
    )
    module = module_cls(activation=cfg.activation, pool=cfg.pool)
    prec = build_fan_in_prior_precision(
        module, cfg.prior_std_weight, cfg.prior_std_bias,
        cfg.fan_in_scaling, dtype=DTYPE, device=DEVICE,
    )
    bm = BayesianModule.build(
        module, likelihood="categorical",
        X=data["X_train"], y=data["y_train"],
        prior_precision=prec, dtype=DTYPE, device=DEVICE,
    )
    assert bm.D == ckpt["D"], f"D mismatch: rebuilt {module_cls.__name__} has D={bm.D}, checkpoint says D={ckpt['D']}"
    x_ref = ckpt["x_ref"].to(dtype=DTYPE, device=DEVICE)
    Sigma_inv = ckpt["Sigma_inv"].to(dtype=DTYPE, device=DEVICE)

    print(f"D = {bm.D}  activation={cfg.activation}  pool={cfg.pool}\n")
    return {
        "name": architecture, "ckpt": ckpt, "cfg": cfg, "bm": bm,
        "x_ref": x_ref, "Sigma_inv": Sigma_inv, "data": data,
    }


refs = {
    "cnn": load_reference(CNN_CKPT_PATH, SEED),
    "lenet5": load_reference(LENET_CKPT_PATH, SEED),
}

## 2. MAP fit quality

Train/test accuracy at `x_ref`, plus a small grid of actual test images
with the model's predicted class and confidence -- a scalar accuracy
number can look "fine" while the model is confidently wrong on
recognizable digits, which the grid makes immediately visible. Run once
per architecture.

In [ ]:
for name, ref in refs.items():
    bm, x_ref, data, ckpt = ref["bm"], ref["x_ref"], ref["data"], ref["ckpt"]
    train_acc = eval_accuracy(bm, x_ref, data["X_train"], data["y_train"])
    test_acc = eval_accuracy(bm, x_ref, data["X_test"], data["y_test"])
    print(f"[{name}]  train_acc = {train_acc:.3f}   test_acc = {test_acc:.3f}")
    print(f"          (checkpoint's own recorded values: train_acc={ckpt.get('train_acc', 'n/a')}, "
          f"test_acc={ckpt.get('test_acc', 'n/a')})")

In [ ]:
@torch.no_grad()
def predict_probs(bm, beta, X):
    logits = torch.func.functional_call(bm.module, bm.param_dict_fn(beta), (X,))
    return torch.softmax(logits, dim=-1)

n_show = 12
for name, ref in refs.items():
    bm, x_ref, data = ref["bm"], ref["x_ref"], ref["data"]
    idx = torch.randperm(data["X_test"].shape[0])[:n_show]
    X_show = data["X_test"][idx]
    y_show = data["y_test"][idx]
    probs = predict_probs(bm, x_ref, X_show)
    pred = probs.argmax(-1)
    conf = probs.max(-1).values

    fig, axes = plt.subplots(2, 6, figsize=(12, 4.5))
    for ax, img, true_y, pred_y, c in zip(axes.flat, X_show.cpu(), y_show.cpu(), pred.cpu(), conf.cpu()):
        ax.imshow(img.squeeze(0), cmap="gray")
        correct = true_y.item() == pred_y.item()
        ax.set_title(f"pred={pred_y.item()} ({c.item():.2f})\ntrue={true_y.item()}",
                     color="#2a7f2a" if correct else "#c0392b", fontsize=9)
        ax.axis("off")
    fig.suptitle(f"[{name}] A sample of test predictions at x_ref", y=1.02)
    fig.tight_layout()
    plt.show()

## 3. Pre-activation saturation check

`print_preactivation_diagnostic` (from `mnist_cnn_grid.py`) reports the
fraction of `conv1`/`conv2` pre-activations with `|z|>2` at `x_ref` and a
handful of prior draws -- `tanh'(2)~0.07`, `tanh'(5)~1.8e-4`, so a large
fraction here means `PRIOR_STD_W` is pushing `tanh` toward its saturated
regime as an artifact of prior scale, not the target's actual curvature.
`LeNet5` uses the same `conv1`/`conv2` attribute names as `CNN`, so this
works unmodified for both architectures. The histogram below shows the
same pre-activation values directly, not just the summary fraction.

In [ ]:
for name, ref in refs.items():
    print(f"[{name}]")
    print_preactivation_diagnostic(ref["bm"], ref["x_ref"], ref["cfg"])
    print()

In [ ]:
for name, ref in refs.items():
    bm, x_ref = ref["bm"], ref["x_ref"]
    hook_outputs = {"conv1": [], "conv2": []}

    def make_hook(hook_name):
        def hook(module, inp, out):
            hook_outputs[hook_name].append(inp[0].detach().flatten())
        return hook

    handles = [
        bm.module.conv1.register_forward_hook(make_hook("conv1")),
        bm.module.conv2.register_forward_hook(make_hook("conv2")),
    ]
    with torch.no_grad():
        torch.func.functional_call(bm.module, bm.param_dict_fn(x_ref), (bm.X[:64],))
    for h in handles:
        h.remove()

    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
    for ax, hook_name in zip(axes, ["conv1", "conv2"]):
        z = hook_outputs[hook_name][0].cpu().numpy()
        ax.hist(z, bins=80, color="#3a6ea5", alpha=0.85)
        ax.axvline(2, color="#c0392b", linestyle="--", linewidth=1)
        ax.axvline(-2, color="#c0392b", linestyle="--", linewidth=1)
        ax.set_title(f"[{name}] {hook_name} pre-activations at x_ref")
        ax.set_xlabel("z")
    fig.tight_layout()
    plt.show()

## 4. `Sigma_inv` inspection, per layer

`Sigma_inv = prior_precision + fisher_diag` (both diagonal, length `D`).
Slicing by `bm.param_dict_fn`'s `{name: shape}` structure and separating
out `Sigma_inv - bm.prior_precision` (the raw empirical-Fisher
contribution) shows whether the Fisher term is actually doing anything
per layer, or whether a layer's precision is coming almost entirely from
the prior. A sane fit typically shows `fc`-type layers (closest to the
loss) with sharper/larger Fisher contribution than early conv layers.
Run once per architecture -- table then bar chart.

In [ ]:
import pandas as pd

sigma_dfs = {}
for name, ref in refs.items():
    bm, Sigma_inv = ref["bm"], ref["Sigma_inv"]
    param_dict = bm.param_dict_fn(Sigma_inv)
    prior_dict = bm.param_dict_fn(bm.prior_precision)

    records = []
    for pname, sigma_slice in param_dict.items():
        prior_slice = prior_dict[pname]
        fisher_slice = sigma_slice - prior_slice
        records.append({
            "param": pname,
            "n": sigma_slice.numel(),
            "Sigma_inv min": sigma_slice.min().item(),
            "Sigma_inv mean": sigma_slice.mean().item(),
            "Sigma_inv max": sigma_slice.max().item(),
            "fisher contrib mean": fisher_slice.mean().item(),
            "fisher contrib max": fisher_slice.max().item(),
        })
    sigma_dfs[name] = pd.DataFrame(records).set_index("param")

print("[cnn]")
display(sigma_dfs["cnn"].style.format(precision=4))
print("[lenet5]")
display(sigma_dfs["lenet5"].style.format(precision=4))

In [ ]:
for name, df in sigma_dfs.items():
    fig, ax = plt.subplots(figsize=(9, 4))
    names = df.index.tolist()
    x = np.arange(len(names))
    width = 0.35

    ax.bar(x - width/2, df["Sigma_inv mean"], width, label="Sigma_inv (mean)", color="#3a6ea5")
    ax.bar(x + width/2, df["fisher contrib mean"], width, label="Fisher contribution (mean)", color="#e08e45")
    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=30, ha="right")
    ax.set_yscale("log")
    ax.set_ylabel("precision (log scale)")
    ax.set_title(f"[{name}] Per-layer Sigma_inv vs. Fisher contribution")
    ax.legend()
    fig.tight_layout()
    plt.show()

## 5. MAP training trajectory

Requires `map_history` in the checkpoint (a list of `{step, loss,
train_acc}` dicts logged during `run_map`). Overlaid rather than
side-by-side, since comparing convergence speed/shape between the two
architectures directly is the useful thing here -- one line per
architecture, per subplot. A checkpoint missing `map_history` is simply
skipped (with a note), rather than blocking the other architecture's
curve from being plotted.

In [ ]:
colors = {"cnn": "#3a6ea5", "lenet5": "#e08e45"}

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
any_history = False
for name, ref in refs.items():
    history = ref["ckpt"].get("map_history")
    if not history:
        print(f"[{name}] No map_history in this checkpoint -- skipping its curve.")
        continue
    any_history = True
    hdf = pd.DataFrame(history)
    axes[0].plot(hdf["step"], hdf["loss"], color=colors[name], marker="o", markersize=3, label=name)
    axes[1].plot(hdf["step"], hdf["train_acc"], color=colors[name], marker="o", markersize=3, label=name)

if any_history:
    axes[0].set_xlabel("step"); axes[0].set_ylabel("energy (loss)"); axes[0].set_title("MAP loss")
    axes[0].legend()
    axes[1].set_xlabel("step"); axes[1].set_ylabel("train accuracy"); axes[1].set_title("MAP train accuracy")
    axes[1].set_ylim(0, 1)
    axes[1].legend()
    fig.tight_layout()
    plt.show()
else:
    plt.close(fig)
    print("No map_history in either checkpoint -- nothing to plot here.")

## 6. Pruning `x_ref` -- how sparse can a sticky cold start be?

The question that actually matters for a sticky sampler run: if we zero out
the smallest-magnitude coordinates of `x_ref` (per-coordinate, relative to
each coordinate's own prior std -- see threshold note below), how much of
`D` can be thrown away before test accuracy actually drops? That tells us
how good a *sparse* `x_ref` can be as the sampler's starting point (cold
start), independent of anything the sampler itself would later do.

Three things below, in order of importance:

1. **6.1 Accuracy vs. sparsity** -- actually zero coordinates below each
   threshold and re-evaluate test accuracy at the pruned point. This is the
   main result: the largest threshold (and corresponding sparsity) at which
   accuracy is still within a small tolerance of the unpruned baseline.
2. **6.2 Per-layer sparsity** -- the same threshold sweep broken out by
   layer, as both a table (a few sample thresholds) and a curve (the full
   sweep), so layer-level differences that the aggregate number in 6.1
   hides are visible directly.
3. **6.3 Sticky prior's assumed sparsity (`kappa`)** -- sanity-checks the
   sticky sampler's own prior belief about sparsity (`prior_inclusion_weight`)
   against what 6.1/6.2 found empirically.

Thresholds throughout are expressed **relative to each coordinate's own
prior std** (`1 / sqrt(prior_precision_i)`), not as one absolute cutoff
across all parameters -- fan-in scaling means weight scales differ hugely
between `conv1` and `fc`-type layers, so an absolute threshold would just
measure which layer happens to have larger weights, not genuine sparsity.

In [ ]:
# --- 6.1 Accuracy vs. sparsity: prune x_ref, re-evaluate test accuracy ---
ACC_DROP_TOLERANCE = 0.01  # max acceptable (baseline_acc - pruned_acc); "minimal cost" cutoff
sweep_multipliers = np.logspace(-3, 0, 60)  # 0.001*std .. 1.0*std
sweep_colors = {"cnn": "#3a6ea5", "lenet5": "#e08e45"}

pruned_accuracy = {}  # name -> {"mult": ..., "frac": ..., "acc": ..., "baseline_acc": ...}
for name, ref in refs.items():
    bm, x_ref, data = ref["bm"], ref["x_ref"], ref["data"]
    prior_std = bm.prior_precision.clamp(min=1e-12).rsqrt()
    baseline_acc = eval_accuracy(bm, x_ref, data["X_test"], data["y_test"])

    fracs, accs = [], []
    for m in sweep_multipliers:
        mask = (x_ref.abs() < m * prior_std)
        x_pruned = torch.where(mask, torch.zeros_like(x_ref), x_ref)
        acc = eval_accuracy(bm, x_pruned, data["X_test"], data["y_test"])
        fracs.append(mask.float().mean().item())
        accs.append(acc)

    fracs = np.array(fracs)
    accs = np.array(accs)
    ok = accs >= (baseline_acc - ACC_DROP_TOLERANCE)
    best_i = np.max(np.nonzero(ok)[0]) if ok.any() else 0
    pruned_accuracy[name] = {
        "mult": sweep_multipliers, "frac": fracs, "acc": accs,
        "baseline_acc": baseline_acc,
        "best_mult": sweep_multipliers[best_i], "best_frac": fracs[best_i], "best_acc": accs[best_i],
    }

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for name, ref in refs.items():
    r = pruned_accuracy[name]
    c = sweep_colors[name]
    axes[0].plot(r["frac"], r["acc"], color=c, linewidth=2, label=f"{name}  (D={ref['bm'].D})")
    axes[0].axhline(r["baseline_acc"], color=c, linestyle=":", linewidth=1, alpha=0.6)
    axes[0].scatter([r["best_frac"]], [r["best_acc"]], color=c, edgecolor="black", zorder=5, s=60)

    axes[1].plot(r["mult"], r["frac"], color=c, linewidth=2, label=f"{name}")
    axes[1].axvline(r["best_mult"], color=c, linestyle=":", linewidth=1, alpha=0.6)

axes[0].set_xlabel("fraction of D pruned (set to zero)")
axes[0].set_ylabel("test accuracy at pruned x_ref")
axes[0].set_title(f"Accuracy vs. sparsity (dotted = unpruned baseline)")
axes[0].legend()

axes[1].set_xscale("log")
axes[1].set_xlabel("threshold, in units of per-coordinate prior std")
axes[1].set_ylabel("fraction of D pruned")
axes[1].set_title("Sparsity of x_ref vs. prune threshold")
axes[1].set_ylim(0, 1)
axes[1].legend()
fig.tight_layout()
plt.show()

In [ ]:
print(f"'Minimal cost' = test accuracy within {ACC_DROP_TOLERANCE:.3f} of the unpruned baseline.\n")
for name, r in pruned_accuracy.items():
    print(f"[{name}]  baseline test_acc = {r['baseline_acc']:.4f}")
    print(f"  best prunable fraction = {r['best_frac']:.3f}  "
          f"(threshold = {r['best_mult']:.4f}*std)  ->  pruned test_acc = {r['best_acc']:.4f}  "
          f"(drop = {r['baseline_acc'] - r['best_acc']:.4f})")
    if name =="cnn":
        print("Leftover number of parameters: ", np.round(19466*(1-pruned_accuracy[name]['best_frac']), 0)+1)
    elif name == "lenet5":
        print("Leftover number of parameters: ", np.round(61706*(1-pruned_accuracy[name]['best_frac']), 0)+1)
    else:
        print()

### 6.2 Per-layer sparsity -- table + sweep together

Same table as before (a few sample thresholds), but now paired directly
with the full sweep curve per layer -- the table alone can't show whether a
layer's sparsity is flat or changes sharply near the sample thresholds;
the curve does.

In [ ]:
THRESHOLD_MULTIPLIERS = [0.01, 0.05, 0.1, 0.2]

sparsity_tables = {}
layer_sweeps = {}  # name -> {param -> frac_array over sweep_multipliers}
for name, ref in refs.items():
    bm, x_ref, cfg = ref["bm"], ref["x_ref"], ref["cfg"]
    prior_std = bm.prior_precision.clamp(min=1e-12).rsqrt()
    x_dict = bm.param_dict_fn(x_ref)
    std_dict = bm.param_dict_fn(prior_std)

    rows = []
    sweeps = {}
    for pname, x_slice in x_dict.items():
        std_slice = std_dict[pname]
        ratio = (x_slice.abs() / std_slice).cpu().numpy()
        row = {"param": pname, "n": x_slice.numel()}
        for m in THRESHOLD_MULTIPLIERS:
            row[f"frac |x|<{m}*std"] = float((ratio < m).mean())
        rows.append(row)
        sweeps[pname] = np.array([(ratio < m).mean() for m in sweep_multipliers])

    overall_ratio = (x_ref.abs() / prior_std).cpu().numpy()
    overall = {"param": "ALL", "n": bm.D}
    for m in THRESHOLD_MULTIPLIERS:
        overall[f"frac |x|<{m}*std"] = float((overall_ratio < m).mean())
    rows.append(overall)
    sweeps["ALL"] = np.array([(overall_ratio < m).mean() for m in sweep_multipliers])

    sparsity_tables[name] = pd.DataFrame(rows).set_index("param")
    layer_sweeps[name] = sweeps

for name in refs:
    print(f"[{name}]")
    display(sparsity_tables[name].style.format(precision=4))

    params = [p for p in layer_sweeps[name] if p != "ALL"]
    cmap = plt.get_cmap("tab10")
    fig, ax = plt.subplots(figsize=(8, 5))
    for i, pname in enumerate(params):
        ax.plot(sweep_multipliers, layer_sweeps[name][pname], color=cmap(i % 10),
                linewidth=1.5, label=pname)
    ax.plot(sweep_multipliers, layer_sweeps[name]["ALL"], color="black",
            linewidth=2.5, linestyle="--", label="ALL")
    ax.set_xscale("log")
    ax.set_xlabel("threshold, in units of per-coordinate prior std")
    ax.set_ylabel("fraction of layer freezable (|x_ref| < threshold)")
    ax.set_title(f"[{name}] Per-layer sparsity vs. threshold")
    ax.set_ylim(0, 1)
    ax.legend(fontsize=8, ncol=2)
    fig.tight_layout()
    plt.show()

### 6.3 Sticky prior's assumed sparsity (`kappa`)

A sticky PDMP sampler (`GridStickyBoomerangSampler`) doesn't just start
from a sparse `x_ref` -- while running, it repeatedly freezes coordinates to
exactly zero and later thaws them, at a **per-coordinate rate `kappa`**.
`kappa` comes from the spike-and-slab prior's assumed sparsity level
`prior_inclusion_weight` (`w`, currently `0.5` by default): a coordinate's
own prior belief is `beta_i ~ w*delta_0 + (1-w)*N(0, sigma_w^2)`, i.e. `w`
is the prior's guess at *what fraction of coordinates are exactly zero*.
`build_kappa_from_inclusion` converts that single scalar `w` into the
per-coordinate `kappa` the sampler actually uses (smaller `kappa` = stickier
= stays frozen longer once it freezes; `kappa` scales with `1/sigma_w`, so
large-prior-std layers like `fc` get a smaller `kappa`, i.e. are stickier,
than small-prior-std layers like `conv1`, for the same `w`).

The check here: does the *assumed* `w` roughly match the *empirically
observed* near-zero fraction from 6.1/6.2 above? If `w` is much larger than
what's empirically observed, the sampler's prior is over-confident about
sparsity and will keep freezing coordinates that the MAP fit says should
stay non-zero (potential accuracy cost); if `w` is much smaller, the
sampler is under-using sparsity the MAP fit already found (potential
wasted effective `D`).

In [ ]:
COMPARE_THRESHOLD = 0.05  # matches one of the THRESHOLD_MULTIPLIERS above

for name, ref in refs.items():
    bm, cfg = ref["bm"], ref["cfg"]
    w = cfg.prior_inclusion_weight
    kappa = build_kappa_from_inclusion(
        bm.module, cfg.prior_std_weight, w, cfg.fan_in_scaling,
        dtype=DTYPE, device=bm.device,
    )
    empirical_frac = sparsity_tables[name].loc["ALL", f"frac |x|<{COMPARE_THRESHOLD}*std"]
    print(f"[{name}]")
    print(f"  prior_inclusion_weight (assumed spike mass, w) = {w:.3f}")
    print(f"  empirical near-zero fraction at {COMPARE_THRESHOLD}*std      = {empirical_frac:.3f}")
    print(f"  -> {'well matched' if abs(w - empirical_frac) < 0.1 else 'MISMATCHED: w and empirical fraction differ by ' + format(abs(w - empirical_frac), '.3f')}")
    print(f"  kappa (weight coords, smaller = stickier): "
          f"min={kappa.min().item():.3e}  mean={kappa.mean().item():.3e}  max={kappa.max().item():.3e}")
    print()

## 7. LeNet5 at maximum sparsity -- layer-by-layer inspection

Section 6.1 found LeNet5's best "minimal cost" prune point at
`threshold = best_mult*std`, leaving only **14005** of `D=61706`
parameters (a **77.3%** sparsity) with just a 0.006 test-accuracy drop.
This section takes that exact pruned `x_ref` and breaks it down per layer
and per parameter-kind (`weight` vs `bias`), to answer: is the pruning
mostly hitting weights, or biases too, and which layers end up with the
most/fewest surviving parameters?

Reuses `pruned_accuracy["lenet5"]["best_mult"]` from 6.1 directly, so this
always reflects whatever threshold 6.1 actually picked -- no threshold is
hardcoded here.

In [ ]:
# --- 7.1 Build the pruned x_ref at LeNet5's best sparsity threshold, per-layer breakdown ---
name = "lenet5"
ref = refs[name]
bm, x_ref = ref["bm"], ref["x_ref"]
best_mult = pruned_accuracy[name]["best_mult"]

prior_std = bm.prior_precision.clamp(min=1e-12).rsqrt()
mask_pruned = (x_ref.abs() < best_mult * prior_std)  # True = zeroed out
x_pruned = torch.where(mask_pruned, torch.zeros_like(x_ref), x_ref)

x_dict = bm.param_dict_fn(x_pruned)
mask_dict = bm.param_dict_fn(mask_pruned)

rows = []
for pname, x_slice in x_dict.items():
    m_slice = mask_dict[pname]
    n_total = m_slice.numel()
    n_pruned = int(m_slice.sum().item())
    n_kept = n_total - n_pruned
    kind = "bias" if pname.endswith(".bias") else "weight"
    rows.append({
        "param": pname,
        "kind": kind,
        "n_total": n_total,
        "n_kept": n_kept,
        "n_pruned": n_pruned,
        "frac_pruned": n_pruned / n_total,
    })

layer_df = pd.DataFrame(rows).set_index("param")
n_total_all = int(layer_df["n_total"].sum())
n_kept_all = int(layer_df["n_kept"].sum())
print(f"[{name}] pruned at threshold = {best_mult:.4f}*std  ->  {n_kept_all}/{n_total_all} params kept "
      f"({n_kept_all / n_total_all:.3%}), {n_total_all - n_kept_all} pruned")
display(layer_df.style.format({"frac_pruned": "{:.3%}"}))

In [ ]:
# --- 7.2 Weights vs. biases -- is pruning mostly hitting weights? ---
kind_df = layer_df.groupby("kind")[["n_total", "n_kept", "n_pruned"]].sum()
kind_df["frac_pruned"] = kind_df["n_pruned"] / kind_df["n_total"]
kind_df["share_of_all_pruned"] = kind_df["n_pruned"] / kind_df["n_pruned"].sum()
print(f"[{name}] pruning breakdown by parameter kind:")
display(kind_df.style.format({"frac_pruned": "{:.3%}", "share_of_all_pruned": "{:.3%}"}))

print(f"\n-> {kind_df.loc['weight', 'n_pruned']} of {n_total_all - n_kept_all} pruned params "
      f"({kind_df.loc['weight', 'share_of_all_pruned']:.1%}) are weights; "
      f"{kind_df.loc['bias', 'n_pruned']} ({kind_df.loc['bias', 'share_of_all_pruned']:.1%}) are biases.")
print(f"   weight-layer prune rate: {kind_df.loc['weight', 'frac_pruned']:.3%}   "
      f"bias-layer prune rate: {kind_df.loc['bias', 'frac_pruned']:.3%}")

In [ ]:
# --- 7.3 Per-layer kept/pruned bar chart, colored by weight vs. bias ---
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

params = layer_df.index.tolist()
x = np.arange(len(params))
kind_color = {"weight": "#3a6ea5", "bias": "#e08e45"}
bar_colors = [kind_color[layer_df.loc[p, "kind"]] for p in params]

axes[0].bar(x, layer_df["n_kept"], color=bar_colors)
axes[0].set_xticks(x)
axes[0].set_xticklabels(params, rotation=30, ha="right")
axes[0].set_ylabel("parameters kept")
axes[0].set_title(f"[{name}] Surviving parameters per layer\n(blue=weight, orange=bias)")
for xi, p in zip(x, params):
    axes[0].text(xi, layer_df.loc[p, "n_kept"], str(layer_df.loc[p, "n_kept"]),
                 ha="center", va="bottom", fontsize=8)

axes[1].bar(x, layer_df["frac_pruned"], color=bar_colors)
axes[1].set_xticks(x)
axes[1].set_xticklabels(params, rotation=30, ha="right")
axes[1].set_ylabel("fraction pruned")
axes[1].set_ylim(0, 1)
axes[1].set_title(f"[{name}] Prune fraction per layer\n(blue=weight, orange=bias)")
for xi, p in zip(x, params):
    axes[1].text(xi, layer_df.loc[p, "frac_pruned"], f"{layer_df.loc[p, 'frac_pruned']:.0%}",
                 ha="center", va="bottom", fontsize=8)

fig.tight_layout()
plt.show()